# Qwen2.5-VL-3B A–G LM-head baseline

This notebook re-extracts frozen Qwen embeddings for the MELD **test split** with a prompt that requires exactly one code: `A=anger`, `B=disgust`, `C=fear`, `D=joy`, `E=neutral`, `F=sadness`, or `G=surprise`. It then applies Qwen's original LM head and predicts the code with the highest next-token logit. No MLP is trained and no test labels influence prediction.

Run from top to bottom with the current `benchmark-code` and `meld-dataset` Kaggle datasets attached. Chunk extraction is resumable.

## 1. Fixed paths and configuration

In [ ]:
from pathlib import Path

CODE = Path("/kaggle/input/datasets/pushkarsingh2005/benchmark-code")
MELD_ROOT = Path("/kaggle/input/datasets/pushkarsingh2005/meld-dataset/MELD.Raw")
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
CODES = list("ABCDEFG")
CODE_TO_EMOTION = dict(zip(CODES, LABELS))

RUN_ROOT = Path("/kaggle/working/experiments/baselines/qwen2_5_vl_3b_lm_head_code/test")
INDEX_DIR = RUN_ROOT / "data"
INDEX_CSV = INDEX_DIR / "meld_test_index.csv"
CHUNKS_DIR = RUN_ROOT / "chunks"
EMBEDDINGS_PT = RUN_ROOT / "embeddings/meld_test.pt"
REPORT_DIR = RUN_ROOT / "reports"
BASELINE_CLEAN_PT = CODE / "experiments/baselines/qwen2_5_vl_3b_shared/embeddings/meld_test_clean.pt"
MLP_METRICS_JSON = CODE / "experiments/baselines/qwen2_5_vl_3b_shared/results/metrics.json"

FPS = 6.0
MIN_FRAMES = 4
MAX_FRAMES = 64
FRAME_SIZE = 224
CHUNK_SIZE = 500
SCORING_BATCH_SIZE = 64

for path in [CODE, MELD_ROOT, BASELINE_CLEAN_PT, MLP_METRICS_JSON]:
    assert path.exists(), f"Missing required input: {path}"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("Run root:", RUN_ROOT)

## 2. Install the tested runtime

In [ ]:
!pip install -q "transformers==4.57.1" "qwen-vl-utils[decord]>=0.0.14" accelerate decord pandas numpy scikit-learn tqdm Pillow opencv-python-headless

In [ ]:
import gc
import json
import os
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import torch
import transformers
from IPython.display import FileLink, display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from tqdm.auto import tqdm
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

assert transformers.__version__ == "4.57.1", "Restart the session after installing Transformers 4.57.1."
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."
DEVICE = torch.device("cuda")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}, {props.total_memory / 2**30:.1f} GiB")

## 3. Rebuild the MELD test index

The index is rebuilt so every absolute video path points to the attached Kaggle MELD dataset.

In [ ]:
prepare_cmd = [
    sys.executable, str(CODE / "scripts/data/prepare_meld.py"),
    "--meld-root", str(MELD_ROOT),
    "--split", "test",
    "--output-dir", str(INDEX_DIR),
]
subprocess.run(prepare_cmd, check=True)
index = pd.read_csv(INDEX_CSV)
assert len(index) > 0 and not index.sample_id.duplicated().any()
assert index.video_path.map(lambda value: Path(value).is_file()).all()
print("Indexed test videos:", len(index))
display(index[["sample_id", "emotion", "utterance"]].head())

## 4. Extract code-prompt embeddings in resumable chunks

Every clip is processed with the same 6 FPS, 4–64 frame, 224 × 224 setup as the shared baseline. The only intended change is `prompt_style=emotion_code`. Each chunk runs in a fresh process to return decoder and model memory to the operating system.

In [ ]:
subprocess_env = os.environ.copy()
subprocess_env["PYTHONUNBUFFERED"] = "1"
extract_cmd = [
    sys.executable, "-u", str(CODE / "scripts/extraction/run_qwen_shared_chunks.py"),
    "--index-csv", str(INDEX_CSV),
    "--chunks-dir", str(CHUNKS_DIR),
    "--chunk-prefix", "test",
    "--chunk-size", str(CHUNK_SIZE),
    "--model-id", MODEL_ID,
    "--fps", str(FPS),
    "--min-frames", str(MIN_FRAMES),
    "--max-frames", str(MAX_FRAMES),
    "--frame-size", str(FRAME_SIZE),
    "--pooling", "last",
    "--prompt-style", "emotion_code",
    "--modality-mode", "video_text",
    "--save-dtype", "float32",
    "--gc-every", "5",
    "--batch-save-every", "25",
    "--retry-errors",
    "--skip-existing-complete",
]
print("Running:", " ".join(extract_cmd), flush=True)
subprocess.run(extract_cmd, check=True, env=subprocess_env)

## 5. Merge and validate extraction

In [ ]:
merge_cmd = [
    sys.executable, str(CODE / "scripts/extraction/merge_embedding_chunks.py"),
    "--chunks-dir", str(CHUNKS_DIR),
    "--output-pt", str(EMBEDDINGS_PT),
    "--pattern", "test_*.pt",
]
subprocess.run(merge_cmd, check=True)
payload = torch.load(EMBEDDINGS_PT, map_location="cpu", weights_only=False)
embeddings = payload["embeddings"].float()
true_ids = payload["labels"].long()
sample_ids = [str(value) for value in payload["sample_ids"]]
metadata = list(payload.get("metadata", []))
errors = list(payload.get("errors", []))
chunk_configs = [entry["config"] for entry in payload.get("config", {}).get("merged_from", [])]

assert chunk_configs and len(set(sample_ids)) == len(sample_ids)
assert embeddings.ndim == 2 and embeddings.shape[1] == 2048
assert embeddings.shape[0] == true_ids.shape[0] == len(sample_ids) == len(metadata)
assert torch.isfinite(embeddings).all()
for config in chunk_configs:
    assert config["model_id"] == MODEL_ID
    assert config["prompt_style"] == "emotion_code"
    assert config["pooling"] == "last" and config["layer"] == -1
    assert config["modality_mode"] == "video_text"
    assert not config["no_generation_prompt"]
    assert float(config["fps"]) == FPS
    assert int(config["min_frames"]) == MIN_FRAMES
    assert int(config["max_frames"]) == MAX_FRAMES
    assert int(config["frame_size"]) == FRAME_SIZE

print("Extracted embeddings:", tuple(embeddings.shape))
print("Extraction errors:", len(errors))
if errors:
    display(pd.DataFrame(errors).head(20))

## 6. Align with the original cleaned test population

The supervised MLP and naive first-token baseline use the cleaned shared test artifact. New embeddings are reordered to exactly those sample IDs. The notebook stops if any reference sample is missing, preventing a partial extraction from appearing comparable.

In [ ]:
baseline_payload = torch.load(BASELINE_CLEAN_PT, map_location="cpu", weights_only=False)
target_ids = [str(value) for value in baseline_payload["sample_ids"]]
new_position = {sample_id: position for position, sample_id in enumerate(sample_ids)}
missing_target_ids = [sample_id for sample_id in target_ids if sample_id not in new_position]
extra_extracted_ids = [sample_id for sample_id in sample_ids if sample_id not in set(target_ids)]
alignment = {
    "indexed_test_videos": len(index),
    "successful_extractions": len(sample_ids),
    "extraction_errors": len(errors),
    "target_clean_sample_count": len(target_ids),
    "missing_target_ids": missing_target_ids,
    "extra_extracted_ids_excluded": extra_extracted_ids,
}
(REPORT_DIR / "sample_alignment.json").write_text(json.dumps(alignment, indent=2), encoding="utf-8")
assert not missing_target_ids, f"Missing {len(missing_target_ids)} baseline samples; retry extraction errors before scoring"
order = torch.tensor([new_position[sample_id] for sample_id in target_ids], dtype=torch.long)
embeddings = embeddings.index_select(0, order)
true_ids = true_ids.index_select(0, order)
metadata = [metadata[position] for position in order.tolist()]
sample_ids = target_ids
assert true_ids.tolist() == baseline_payload["labels"].long().tolist()
print(json.dumps({key: value for key, value in alignment.items() if not key.endswith("ids") and not key.endswith("excluded")}, indent=2))
print("Aligned samples to score:", len(sample_ids))

## 7. Verify A–G contextual tokenization

In [ ]:
sys.path.insert(0, str(CODE))
from scripts.extraction.extract_qwen_shared_embeddings import make_prompt

processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
probe_prompt = make_prompt("Tokenization probe.", "emotion_code", "video_text")
probe_messages = [{"role": "user", "content": [{"type": "text", "text": probe_prompt}]}]
assistant_prefix = processor.apply_chat_template(probe_messages, tokenize=False, add_generation_prompt=True)
prefix_ids = tokenizer(assistant_prefix, add_special_tokens=False)["input_ids"]
token_rows = []
for code, emotion in zip(CODES, LABELS):
    full_ids = tokenizer(assistant_prefix + code, add_special_tokens=False)["input_ids"]
    assert full_ids[:len(prefix_ids)] == prefix_ids, f"Tokenizer boundary changed for {code}"
    candidate_ids = full_ids[len(prefix_ids):]
    token_rows.append({
        "code": code, "emotion": emotion, "token_count": len(candidate_ids),
        "token_ids": json.dumps(candidate_ids),
        "tokens": json.dumps(tokenizer.convert_ids_to_tokens(candidate_ids), ensure_ascii=False),
        "token_id": int(candidate_ids[0]) if candidate_ids else None,
    })
token_audit = pd.DataFrame(token_rows)
assert (token_audit.token_count == 1).all(), token_audit
code_token_ids = token_audit.token_id.astype(int).tolist()
assert len(set(code_token_ids)) == len(CODES), "A-G token IDs are not distinct"
token_audit.to_csv(REPORT_DIR / "code_tokenization.csv", index=False)
display(token_audit)

## 8. Retain Qwen's LM head and score the aligned embeddings

The checkpoint is loaded on CPU once more from the local Hugging Face cache. Only the frozen LM head is retained and moved to GPU; the vision encoder and multimodal transformer are deleted before scoring.

In [ ]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map={"": "cpu"}, low_cpu_mem_usage=True
)
assert model.lm_head.in_features == embeddings.shape[1]
assert max(code_token_ids) < model.lm_head.out_features
model_revision = getattr(model.config, "_commit_hash", None)
lm_head = model.lm_head
del model
gc.collect()
torch.cuda.empty_cache()
lm_head = lm_head.to(DEVICE).eval()
token_ids_tensor = torch.tensor(code_token_ids, dtype=torch.long, device=DEVICE)

selected_logits_parts, raw_prob_parts, restricted_prob_parts = [], [], []
vocab_top_id_parts, vocab_top_prob_parts = [], []
with torch.inference_mode():
    for start in tqdm(range(0, len(embeddings), SCORING_BATCH_SIZE), desc="LM-head scoring"):
        batch = embeddings[start:start + SCORING_BATCH_SIZE].to(DEVICE, dtype=lm_head.weight.dtype)
        vocabulary_logits = lm_head(batch).float()
        selected_logits = vocabulary_logits.index_select(1, token_ids_tensor)
        vocabulary_log_probs = torch.log_softmax(vocabulary_logits, dim=-1)
        raw_probs = vocabulary_log_probs.index_select(1, token_ids_tensor).exp()
        restricted_probs = torch.softmax(selected_logits, dim=-1)
        top_logits, top_ids = vocabulary_logits.max(dim=-1)
        top_probs = (top_logits - torch.logsumexp(vocabulary_logits, dim=-1)).exp()
        selected_logits_parts.append(selected_logits.cpu())
        raw_prob_parts.append(raw_probs.cpu())
        restricted_prob_parts.append(restricted_probs.cpu())
        vocab_top_id_parts.append(top_ids.cpu())
        vocab_top_prob_parts.append(top_probs.cpu())
        del batch, vocabulary_logits, vocabulary_log_probs

selected_logits = torch.cat(selected_logits_parts).numpy()
raw_probs = torch.cat(raw_prob_parts).numpy()
restricted_probs = torch.cat(restricted_prob_parts).numpy()
vocab_top_ids = torch.cat(vocab_top_id_parts).numpy()
vocab_top_probs = torch.cat(vocab_top_prob_parts).numpy()
pred_ids = restricted_probs.argmax(axis=1)
sorted_probs = np.sort(restricted_probs, axis=1)
confidence = sorted_probs[:, -1]
margin = sorted_probs[:, -1] - sorted_probs[:, -2]
entropy = -(restricted_probs * np.log(np.clip(restricted_probs, 1e-12, 1.0))).sum(axis=1)
candidate_mass = raw_probs.sum(axis=1)

## 9. Export predictions, metrics, and comparison

In [ ]:
true_np = true_ids.numpy()
predictions = pd.DataFrame({
    "sample_id": sample_ids,
    "true_id": true_np,
    "true_emotion": [LABELS[index] for index in true_np],
    "predicted_id": pred_ids,
    "predicted_code": [CODES[index] for index in pred_ids],
    "predicted_emotion": [LABELS[index] for index in pred_ids],
    "correct": pred_ids == true_np,
    "restricted_confidence": confidence,
    "restricted_margin": margin,
    "restricted_entropy": entropy,
    "candidate_probability_mass": candidate_mass,
    "vocab_top_token_id": vocab_top_ids,
    "vocab_top_token": tokenizer.convert_ids_to_tokens(vocab_top_ids.tolist()),
    "vocab_top_probability": vocab_top_probs,
})
for column in ["dialogue_id", "utterance_id", "utterance", "video_file"]:
    predictions[column] = [row.get(column) for row in metadata]
for position, (code, emotion) in enumerate(zip(CODES, LABELS)):
    suffix = f"{code}_{emotion}"
    predictions[f"logit_{suffix}"] = selected_logits[:, position]
    predictions[f"raw_vocab_prob_{suffix}"] = raw_probs[:, position]
    predictions[f"restricted_prob_{suffix}"] = restricted_probs[:, position]

cm = confusion_matrix(true_np, pred_ids, labels=list(range(len(LABELS))))
report = classification_report(true_np, pred_ids, labels=list(range(len(LABELS))), target_names=LABELS, output_dict=True, zero_division=0)
majority_id = int(np.bincount(true_np, minlength=len(LABELS)).argmax())
majority_accuracy = float((true_np == majority_id).mean())
top_is_code_rate = float(np.isin(vocab_top_ids, np.array(code_token_ids)).mean())
mlp_metrics = json.loads(MLP_METRICS_JSON.read_text())["test"]
metrics = {
    "experiment": "qwen2_5_vl_3b_lm_head_code",
    "split": "test", "model_id": MODEL_ID, "model_revision": model_revision,
    "sample_count": len(predictions),
    "scoring_rule": "argmax over seven contextual single-token A-G logits",
    "accuracy": float(accuracy_score(true_np, pred_ids)),
    "macro_f1": float(f1_score(true_np, pred_ids, average="macro", zero_division=0)),
    "weighted_f1": float(f1_score(true_np, pred_ids, average="weighted", zero_division=0)),
    "majority_label": LABELS[majority_id], "majority_accuracy": majority_accuracy,
    "accuracy_minus_majority": float(accuracy_score(true_np, pred_ids) - majority_accuracy),
    "unrestricted_top_token_is_code_rate": top_is_code_rate,
    "mean_restricted_confidence": float(confidence.mean()),
    "mean_restricted_margin": float(margin.mean()),
    "mean_restricted_entropy": float(entropy.mean()),
    "mean_candidate_probability_mass": float(candidate_mass.mean()),
    "confusion_matrix": cm.tolist(), "classification_report": report,
    "label_names": LABELS, "code_mapping": CODE_TO_EMOTION,
    "code_token_ids": dict(zip(CODES, code_token_ids)),
    "reference_supervised_mlp": {key: float(mlp_metrics[key]) for key in ["accuracy", "macro_f1", "weighted_f1"]},
}
predictions.to_csv(REPORT_DIR / "test_predictions.csv", index=False)
pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS], columns=[f"pred_{x}" for x in LABELS]).to_csv(REPORT_DIR / "test_confusion_matrix.csv")
pd.DataFrame(report).T.to_csv(REPORT_DIR / "test_classification_report.csv")
(REPORT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
manifest = {
    "index_csv": str(INDEX_CSV), "embeddings_path": str(EMBEDDINGS_PT),
    "embedding_shape": list(embeddings.shape), "transformers_version": transformers.__version__,
    "torch_version": torch.__version__, "fps": FPS, "min_frames": MIN_FRAMES,
    "max_frames": MAX_FRAMES, "frame_size": FRAME_SIZE, "prompt_style": "emotion_code",
    "pooling": "last", "layer": -1, "scoring_batch_size": SCORING_BATCH_SIZE,
}
(REPORT_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps({key: metrics[key] for key in ["accuracy", "macro_f1", "weighted_f1", "majority_accuracy", "accuracy_minus_majority", "unrestricted_top_token_is_code_rate"]}, indent=2))

## 10. Inspect and download

In [ ]:
display(pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS], columns=[f"pred_{x}" for x in LABELS]))
display(pd.DataFrame(report).T)
display(predictions.head(10))
archive = shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT.parent, base_dir=RUN_ROOT.name)
print("Download bundle:", archive)
display(FileLink(os.path.relpath(archive, Path.cwd())))